# Semana 9 — Hugging Face por Dentro: Tokenizers, Hidden States e Generation
**Fase 2: Deep Learning e o Ecossistema Hugging Face** · Gabarito (notebook resolvido)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/applied-ai-architecture/blob/main/02-Phase-2-Deep-Learning/notebooks/semana_09_gabarito.ipynb)

Leitura de conceito desta semana: [Semana-09-Conceitos.md](https://github.com/NetoRibeiro/applied-ai-architecture/blob/main/02-Phase-2-Deep-Learning/docs/Semana-09-Conceitos.md)

Use este gabarito **depois** de tentar o notebook de exercícios. Compare a sua solução, não copie.

> **GPU:** em *Ambiente de execução → Alterar o tipo de ambiente de execução*, selecione **T4 GPU** antes de começar.

In [ ]:
import torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
device = "cuda" if torch.cuda.is_available() else "cpu"; torch.manual_seed(0); print(device)
MODELO_LM = "distilgpt2"
MODELO_EMB = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

## Passo 1 — Tokenizer por dentro

In [ ]:
tok = AutoTokenizer.from_pretrained(MODELO_LM)
for frase in ["The architecture of large language models", "A arquitetura de grandes modelos de linguagem"]:
    ids = tok(frase)["input_ids"]
    pedacos = tok.convert_ids_to_tokens(ids)
    print(f"{len(frase.split())} palavras -> {len(ids)} tokens: {pedacos}")
print("vocabulário:", tok.vocab_size, "| pad_token:", tok.pad_token, "| eos_token:", tok.eos_token)

Por que a frase em português consome mais tokens? O que isso implica em custo?

## Passo 2 — Carregar o modelo e ler a config

In [ ]:
lm = AutoModelForCausalLM.from_pretrained(MODELO_LM).to(device).eval()
c = lm.config
n_params = sum(p.numel() for p in lm.parameters())
print(f"d={c.n_embd} camadas={c.n_layer} cabeças={c.n_head} contexto={c.n_positions} vocab={c.vocab_size}")
estimativa = c.vocab_size * c.n_embd + 12 * c.n_layer * c.n_embd ** 2
mem_fp32_mb = n_params * 4 / 1e6
print(f"parâmetros: {n_params/1e6:.1f}M | estimativa: {estimativa/1e6:.1f}M | memória fp32 ≈ {mem_fp32_mb:.0f} MB")

## Passo 3 — Hidden states e logits

In [ ]:
entrada = tok("The capital of France is", return_tensors="pt").to(device)
with torch.no_grad():
    out = lm(**entrada, output_hidden_states=True)
print("nº de hidden states:", len(out.hidden_states), "| shape de cada um:", tuple(out.hidden_states[-1].shape))
print("logits:", tuple(out.logits.shape))

## Passo 4 — O próximo token, na mão
Pegue os logits da **última posição**, aplique softmax e liste os 5 tokens mais prováveis.

In [ ]:
probs = F.softmax(out.logits[0, -1], dim=-1)
top = torch.topk(probs, 5)
for p, i in zip(top.values, top.indices):
    print(f"{p.item():.3f}  {tok.decode(i)!r}")

## Passo 5 — Greedy decoding manual vs. `generate()`
Escreva o loop: a cada passo, escolha o `argmax` da última posição e anexe. Compare com `generate(do_sample=False)`.

In [ ]:
ids = entrada["input_ids"]
with torch.no_grad():
    for _ in range(12):
        prox = lm(ids).logits[:, -1].argmax(dim=-1, keepdim=True)
        ids = torch.cat([ids, prox], dim=1)
manual = tok.decode(ids[0])
oficial = tok.decode(lm.generate(**entrada, max_new_tokens=12, do_sample=False, pad_token_id=tok.eos_token_id)[0])
print("manual :", manual, "\noficial:", oficial); assert manual == oficial

## Passo 6 — Estratégias de decodificação

In [ ]:
prompt = tok("In the future, artificial intelligence will", return_tensors="pt").to(device)
configs = {
    "greedy": dict(do_sample=False),
    "temperatura 0.7": dict(do_sample=True, temperature=0.7),
    "top-k 40": dict(do_sample=True, top_k=40),
    "top-p 0.9": dict(do_sample=True, top_p=0.9, top_k=0),
}
for nome, kw in configs.items():
    torch.manual_seed(0)
    saida = lm.generate(**prompt, max_new_tokens=30, pad_token_id=tok.eos_token_id, repetition_penalty=1.2, **kw)
    print(f"\n[{nome}] {tok.decode(saida[0], skip_special_tokens=True)}")

## Passo 7 — Batch generation: o padding side importa

In [ ]:
tok.pad_token = tok.eos_token
prompts = ["Hello", "The three primary colors are"]
for lado in ("right", "left"):
    tok.padding_side = lado
    lote = tok(prompts, return_tensors="pt", padding=True).to(device)
    saida = lm.generate(**lote, max_new_tokens=8, do_sample=False, pad_token_id=tok.eos_token_id)
    print(f"padding {lado:5s}:", [tok.decode(s, skip_special_tokens=True) for s in saida])

Qual dos dois lados produz continuações coerentes para o prompt curto? Por quê?

## Passo 8 — Embeddings de frase com um encoder
Implemente o **mean pooling**: média dos vetores dos tokens reais (use a `attention_mask`), depois normalize.

In [ ]:
tok_e = AutoTokenizer.from_pretrained(MODELO_EMB)
enc = AutoModel.from_pretrained(MODELO_EMB).to(device).eval()

@torch.no_grad()
def embed(frases):
    lote = tok_e(frases, padding=True, truncation=True, return_tensors="pt").to(device)
    h = enc(**lote).last_hidden_state                      # (B, T, d)
    mascara = lote["attention_mask"].unsqueeze(-1).float()  # (B, T, 1)
    media = (h * mascara).sum(dim=1) / mascara.sum(dim=1)
    return F.normalize(media, dim=-1)

frases = ["Como peço reembolso de uma viagem?", "Qual o processo para reaver despesas de deslocamento?",
          "O servidor de produção caiu ontem à noite.", "How do I get my travel expenses refunded?"]
E = embed(frases); print("shape:", tuple(E.shape))
print((E @ E.T).cpu().numpy().round(2))

As frases 1, 2 e 4 dizem a mesma coisa (a 4 em inglês). O que a matriz mostra? Este é o dot product da Semana 2 operando sobre significado.

## Passo 9 — Precision e memória
Recarregue o modelo em `float16` (apenas com GPU) e compare a memória.

In [ ]:
def mem_mb(m): return sum(p.numel() * p.element_size() for p in m.parameters()) / 1e6
print(f"fp32: {mem_mb(lm):.0f} MB")
if device == "cuda":
    lm16 = AutoModelForCausalLM.from_pretrained(MODELO_LM, torch_dtype=torch.float16).to(device)
    print(f"fp16: {mem_mb(lm16):.0f} MB | GPU alocada: {torch.cuda.memory_allocated()/1e6:.0f} MB")

## Teste Prático de Entrevista — Nucleus Sampling (Temperature + Top-p)
Um entrevistador pede: *"Implemente `probs_top_p(logits, temperature=1.0, top_p=0.9)` retornando a distribuição de probabilidade final após temperatura e corte nucleus (mantendo os tokens de maior probabilidade cuja soma acumulada atinge `top_p`, sempre mantendo ao menos 1 token, e renormalizando para somar `1.0`). Se `temperature == 0`, retorne one-hot no argmax."* 

In [ ]:
import math


def probs_top_p(logits, temperature=1.0, top_p=0.9):
    """Aplica temperature scaling + top-p filtering e retorna probabilidades normalizadas."""
    if temperature == 0.0:
        best = max(range(len(logits)), key=lambda i: logits[i])
        return [1.0 if i == best else 0.0 for i in range(len(logits))]

    scaled = [l / temperature for l in logits]
    m = max(scaled)
    exps = [math.exp(l - m) for l in scaled]
    total = sum(exps)
    probs = [e / total for e in exps]

    order = sorted(range(len(probs)), key=lambda i: probs[i], reverse=True)
    kept = []
    cum = 0.0
    for idx in order:
        kept.append(idx)
        cum += probs[idx]
        if cum >= top_p:
            break

    kept_set = set(kept)
    filtered = [probs[i] if i in kept_set else 0.0 for i in range(len(probs))]
    norm = sum(filtered)
    return [p / norm for p in filtered]


p_greedy = probs_top_p([1.0, 3.0, 2.0], temperature=0.0, top_p=0.9)
assert p_greedy == [0.0, 1.0, 0.0]
p_nuc = probs_top_p([5.0, 2.0, -5.0], temperature=1.0, top_p=0.9)
assert abs(sum(p_nuc) - 1.0) < 1e-9
assert p_nuc[2] == 0.0 and p_nuc[0] > 0.9
print("OK! Top-p probs:", p_nuc)